In [1]:
# sample 1 out of every 16 frames from each video and store on disk

import os
import cv2
import math
import warnings
from tqdm import tqdm

warnings.filterwarnings("ignore")

VIDEO_ROOT = "/kaggle/input/datasets/shubhamnitc/ucf-crime-test-split/ucf-crime_test_split"
SAVE_ROOT = "/kaggle/working/ucf_crime_test_frames"
STRIDE = 16

os.makedirs(SAVE_ROOT, exist_ok=True)

video_files = sorted(os.listdir(VIDEO_ROOT))

for video_file in tqdm(video_files):

    video_path = os.path.join(VIDEO_ROOT, video_file)
    video_name = os.path.splitext(video_file)[0]
    save_dir = os.path.join(SAVE_ROOT, video_name)

    os.makedirs(save_dir, exist_ok=True)

    existing_frames = [f for f in os.listdir(save_dir) if f.endswith(".jpg")]
    existing_count = len(existing_frames)

    cap = cv2.VideoCapture(video_path)
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    cap.release()

    expected_count = math.ceil(total_frames / STRIDE)

    if existing_count == expected_count:
        continue

    cap = cv2.VideoCapture(video_path)

    for frame_idx in range(total_frames):

        ret, frame = cap.read()
        if not ret:
            break

        if frame_idx % STRIDE != 0:
            continue

        save_path = os.path.join(save_dir, f"{frame_idx:06d}.jpg")

        if os.path.exists(save_path):
            continue

        frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        cv2.imwrite(save_path, frame_rgb)

    cap.release()

print("Frame sampling complete.")

100%|██████████| 290/290 [05:15<00:00,  1.09s/it]

Frame sampling complete.


In [2]:
# check available GPUs and their memory

import torch
import warnings

warnings.filterwarnings("ignore")

gpu_count = torch.cuda.device_count()

print("Total GPUs:", gpu_count)

for i in range(gpu_count):
    print(f"\nGPU {i}:")
    print(" Name:", torch.cuda.get_device_name(i))
    print(" Total Memory (GB):", torch.cuda.get_device_properties(i).total_memory / 1024**3)

Total GPUs: 2

GPU 0:
 Name: Tesla T4
 Total Memory (GB): 14.56317138671875

GPU 1:
 Name: Tesla T4
 Total Memory (GB): 14.56317138671875


In [3]:
# define cuda:0 and cuda:1 for controlled GPU usage

import torch
import warnings

warnings.filterwarnings("ignore")

device_0 = torch.device("cuda:0")
device_1 = torch.device("cuda:1")

print("device_0:", device_0)
print("device_1:", device_1)

device_0: cuda:0
device_1: cuda:1


In [4]:
# install CLIP

import warnings
warnings.filterwarnings("ignore")

!pip install git+https://github.com/openai/CLIP.git -q

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 2.1 MB/s eta 0:00:00


In [5]:
# load frozen CLIP ViT-L/14 model on device_0 for visual embeddings

import torch
import clip
import warnings

warnings.filterwarnings("ignore")

torch.cuda.set_device(device_0)

model, preprocess = clip.load("ViT-L/14", device=device_0)
model.eval()

for p in model.parameters():
    p.requires_grad = False

print("CLIP loaded on:", device_0)
print("Memory allocated (MB):", torch.cuda.memory_allocated(device_0) / 1024**2)

100%|███████████████████████████████████████| 890M/890M [00:10<00:00, 93.0MiB/s]


CLIP loaded on: cuda:0
Memory allocated (MB): 899.72216796875


In [6]:
# compute L2-normalized CLIP embeddings for one sampled video

import os
import torch
from PIL import Image
import warnings

warnings.filterwarnings("ignore")

FRAME_ROOT = "/kaggle/working/ucf_crime_test_frames"
video_name = "Abuse030_x264"
frame_dir = os.path.join(FRAME_ROOT, video_name)

frame_files = sorted([f for f in os.listdir(frame_dir) if f.endswith(".jpg")])

images = []
for f in frame_files:
    img = Image.open(os.path.join(frame_dir, f)).convert("RGB")
    images.append(preprocess(img))

images = torch.stack(images).to(device_0)

with torch.no_grad():
    features = model.encode_image(images)
    features = features / features.norm(dim=-1, keepdim=True)

print("Frames:", len(frame_files))
print("Embedding shape:", features.shape)
print("Norm:", features[0].norm().item())

Frames: 97
Embedding shape: torch.Size([97, 768])
Norm: 0.99951171875


In [7]:
# generate and save L2-normalized CLIP embeddings for all sampled videos

import os
import torch
import numpy as np
from PIL import Image
from tqdm import tqdm
import warnings

warnings.filterwarnings("ignore")

FRAME_ROOT = "/kaggle/working/ucf_crime_test_frames"
SAVE_ROOT = "/kaggle/working/ucf_crime_test_clip_embeddings"

os.makedirs(SAVE_ROOT, exist_ok=True)

video_names = sorted(os.listdir(FRAME_ROOT))
batch_size = 32

for video_name in tqdm(video_names):

    save_path = os.path.join(SAVE_ROOT, f"{video_name}.npy")

    if os.path.exists(save_path):
        continue

    frame_dir = os.path.join(FRAME_ROOT, video_name)
    frame_files = sorted([f for f in os.listdir(frame_dir) if f.endswith(".jpg")])

    all_embeddings = []

    with torch.no_grad():
        for i in range(0, len(frame_files), batch_size):

            batch_files = frame_files[i:i+batch_size]
            images = []

            for f in batch_files:
                img = Image.open(os.path.join(frame_dir, f)).convert("RGB")
                images.append(preprocess(img))

            images = torch.stack(images).to(device_0)

            features = model.encode_image(images)
            features = features / features.norm(dim=-1, keepdim=True)

            all_embeddings.append(features.cpu())

    embeddings = torch.cat(all_embeddings, dim=0).numpy()
    np.save(save_path, embeddings)

print("All embeddings complete.")

100%|██████████| 290/290 [18:43<00:00,  3.88s/it]

All embeddings complete.


In [8]:
# verify saved CLIP embedding files for count, shape, and normalization

import os
import numpy as np
import random
import warnings

warnings.filterwarnings("ignore")

EMBED_ROOT = "/kaggle/working/ucf_crime_test_clip_embeddings"

files = sorted([f for f in os.listdir(EMBED_ROOT) if f.endswith(".npy")])

print("Total embedding files:", len(files))

sample_files = random.sample(files, min(5, len(files)))

for f in sample_files:
    path = os.path.join(EMBED_ROOT, f)
    arr = np.load(path)

    print(f"\nFile: {f}")
    print(" Shape:", arr.shape)
    print(" Dim check:", arr.shape[1] == 768)
    print(" First vector norm:", np.linalg.norm(arr[0]))

Total embedding files: 290

File: Shoplifting029_x264.npy
 Shape: (136, 768)
 Dim check: True
 First vector norm: 1.0

File: Normal_Videos_889_x264.npy
 Shape: (20, 768)
 Dim check: True
 First vector norm: 1.0

File: Normal_Videos_903_x264.npy
 Shape: (50, 768)
 Dim check: True
 First vector norm: 1.0

File: Normal_Videos_934_x264.npy
 Shape: (111, 768)
 Dim check: True
 First vector norm: 1.0

File: Normal_Videos_453_x264.npy
 Shape: (333, 768)
 Dim check: True
 First vector norm: 0.9995
